<a href="https://colab.research.google.com/github/Ali-datasmith/data-science-foundations/blob/main/duckdb%2Bsql/duckdb%2Bsql_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦆 DuckDB + SQL for Data Science
**Author:** Muhammad Ali Rajput
**Date:** April 2026
**Tools:** Python 3.12 | DuckDB | Polars | Plotly | Rich | Google Colab

---

I already knew Polars well before starting this. Zero SQL background. This notebook is my personal record of how I learned DuckDB in one day — by mapping every SQL concept directly to the Polars code I already understood.

The approach that worked for me: **see Polars first, then see the SQL equivalent doing the exact same thing.** Once you realize they're the same logic with different syntax, it clicks fast.

---

## 📚 Table of Contents

| # | Topic | What it covers |
|---|-------|----------------|
| 1 | Setup & Dataset | Install libraries, build our working DataFrame |
| 2 | `duckdb.connect()` | Creating a database connection in memory |
| 3 | `SELECT` + `WHERE` | Filtering rows and picking columns — Polars vs SQL |
| 4 | Practice: WHERE filter | First hands-on exercise |
| 5 | `GROUP BY` + `SUM` | Aggregating data by category |
| 6 | Practice: GROUP BY + AVG | Second hands-on exercise |
| 7 | Hybrid Workflow | Querying Polars DataFrames directly with DuckDB |
| 8 | Final Challenge | DuckDB query → Plotly chart (the whole pipeline) |
| 9 | What We Covered | Summary and key takeaways |

---

## 1. Setup & Dataset

First things first — install DuckDB and Polars, then import everything we need.

The dataset is a simple product sales table. Five products, two categories, with price and units sold. Small enough to see clearly, realistic enough to actually learn from. This same data is used throughout the entire notebook.

In [ ]:
!pip install duckdb polars -q

import duckdb
import polars as pl
from rich import print

# Our master dataset
df = pl.DataFrame({
    "product": ["Laptop", "Mouse", "Monitor", "Keyboard", "Headset"],
    "category": ["Electronics", "Accessories", "Electronics", "Accessories", "Electronics"],
    "price": [1200, 25, 300, 45, 85],
    "units_sold": [10, 50, 20, 30, 15]
})

**Quick look at what we're working with.** Five rows, four columns — product name, category, price, and how many units were sold.

In [ ]:
print(df.head())

---
## 2. `duckdb.connect()` — Opening a database engine

Before running any SQL, you need a *connection*. Think of it as turning on the engine before driving.

`':memory:'` means the database lives in RAM — nothing is saved to disk. For our learning purposes and for most data projects, this is exactly what we want. Fast, clean, no leftover files.

In [ ]:
# Create an in-memory database connection
con = duckdb.connect(database=':memory:')
print("Connected to DuckDB!")

**Practice task:** Create your own connection called `my_con` — same thing, just your own variable name. This is the connection we'll use for all the SQL queries below.

In [ ]:
my_con = duckdb.connect(database=':memory:')
print("\n[yellow]Connected to DuckDB!")

---
## 3. `SELECT` + `WHERE` — Filtering rows and picking columns

This is where SQL starts to feel familiar if you know Polars.

- **Polars way:** `.filter()` to keep rows, `.select()` to pick columns
- **SQL way:** `WHERE` to keep rows, `SELECT` to pick columns

Same result. The logic is identical — just written differently.

One important thing I had to remember: in SQL, `=` is used for comparison (not `==` like in Python). That's the most common beginner mistake.

In [ ]:
# Filtering for expensive items and picking specific columns
result_pl = df.filter(pl.col("price") > 100).select(["product", "price"])
print(result_pl)

Same operation, now with DuckDB. Notice `.df()` at the end — that converts the SQL result back into a DataFrame so we can keep working with it in Python.

In [ ]:
# Running a SQL query string
# .df() at the end converts the result back into a Polars-friendly format (DataFrame)
result_sql = my_con.execute("SELECT product, price FROM df WHERE price > 100").df()
print(result_sql)

---
## 4. Practice — `WHERE` filter

**Task:** Filter the data to show only `Accessories`, and return just the `product` and `units_sold` columns.

Try it in Polars first, then write the same thing as a SQL query.

In [ ]:
result_pl = df.filter(pl.col('category')=='Accessories').select(['product','units_sold'])
print(result_pl)

Now the SQL version of the exact same filter. Notice `category = 'Accessories'` uses single quotes around the string value — that's standard SQL.

In [ ]:
result_sql = my_con.execute("SELECT product,units_sold FROM df WHERE category = 'Accessories'").df()
print(result_sql)

---
## 5. `GROUP BY` + `SUM` — Aggregating data by category

This is one of the most useful things in any data project — collapsing many rows into summary numbers.

- **Polars way:** `.group_by("category").agg(pl.col("units_sold").sum())`
- **SQL way:** `GROUP BY category` with `SUM(units_sold)` inside `SELECT`

The interesting difference: in SQL, you write the math (`SUM`, `AVG`, `COUNT`) inside the `SELECT` statement. In Polars, you write it inside `.agg()`. Same idea, different place.

In [ ]:
# Grouping by category and summing units
result_pl = df.group_by("category").agg(pl.col("units_sold").sum().alias("total_units"))
print(result_pl)

SQL version of the same aggregation. The triple-quote string `"""` is just a clean way to write multi-line SQL in Python — makes it easier to read.

In [ ]:
# In SQL, the column name and the math happen inside the SELECT
query = """
SELECT category, SUM(units_sold) AS total_units
FROM df
GROUP BY category
"""
result_sql = my_con.execute(query).df()
print(result_sql)

---
## 6. Practice — `GROUP BY` + `AVG`

**Task:** Instead of summing units, calculate the **average price** per category.

`AVG()` in SQL is the same as `.mean()` in Polars.

In [ ]:
result_pl = df.group_by(pl.col('category')).agg(pl.col('price').mean().alias('average_price'))
print(result_pl)

SQL version using `AVG()`. The result should match the Polars output above exactly.

In [ ]:
query = '''
SELECT category,AVG(price) as average_price
FROM df
GROUP BY category
'''
result_sql = my_con.execute(query).df()
print(result_sql)

A quick sanity check — selecting just the product names to make sure we understand what `.select()` does on its own.

In [ ]:
df.select("product")

---
## 7. Hybrid Workflow — Querying Polars DataFrames directly with DuckDB

This is the most powerful thing about DuckDB in Python: **it can see your Polars DataFrames as if they were database tables** — without any import or conversion step.

You just create a DataFrame with a variable name, and DuckDB automatically makes it available as a table with that same name. This is called "Replacement Scans" and it's what makes DuckDB perfect for our kind of work.

So `my_con.execute("SELECT * FROM df")` — `df` there refers to the actual Python variable `df` we created at the top.

In [ ]:
# Even if we create a NEW polars dataframe...
discount_df = df.with_columns(pl.col("price") * 0.9)

# DuckDB sees 'discount_df' immediately!
query = "SELECT * FROM discount_df WHERE price < 50"
print(my_con.execute(query).df())

Another example — I created a filtered Polars DataFrame called `high_units`, then immediately queried it with DuckDB. No extra steps needed.

In [ ]:
high_units = df.filter(pl.col('units_sold')>25)
print(high_units.select('product'))

DuckDB sees `high_units` automatically. This is the workflow we'll use in Project 1 — the user uploads a CSV, Polars reads it, DuckDB queries it, Plotly charts it.

In [ ]:
query = '''
SELECT product FROM high_units WHERE units_sold > 25
'''
print(my_con.execute(query).df())

---
## 8. Final Challenge — Full Pipeline: DuckDB → Plotly

This is the real-world workflow. A SQL query prepares the data, then Plotly turns it into a chart.

**Challenge part 1:** Use DuckDB to query `df`, order by `units_sold` descending, and pass the result directly into a Plotly bar chart.

In [ ]:
import plotly.express as px

# 1. The SQL Query
query = """
SELECT product, units_sold
FROM df
ORDER BY units_sold DESC
"""
chart_data = my_con.execute(query).df()

# 2. The Plotly Chart
fig = px.bar(chart_data, x='product', y='units_sold', title="Units Sold by Product")
fig.show()

**Challenge part 2 (Polars version):** Same revenue chart using Polars Lazy API instead of SQL. Both approaches produce the same result — knowing both gives you flexibility.

In [ ]:
import plotly.express as px
result = (
    df.lazy()
    .with_columns((pl.col('price')*pl.col('units_sold'))
    .alias('total_revenue'))
    .group_by('category').agg(pl.col('total_revenue').sum().alias('Total_Amount'))
    .collect()
)
fig = px.bar(result,x='category',y='Total_Amount',color='category',template='plotly_dark',title='Revenue Per Category')
fig.show()

**Challenge part 3 (DuckDB version):** Same revenue calculation but using pure SQL. Notice `SUM(price*units_sold)` — DuckDB can do math directly inside the query without needing a separate column.

In [ ]:
query = '''
SELECT category,SUM(price*units_sold) as Total_Revenue
FROM df
GROUP BY category
ORDER BY total_revenue DESC
'''
result = my_con.execute(query).df()
fig = px.bar(result,x='category',y='Total_Revenue',title='Revenue Per Category',template='plotly_dark',color='category')
fig.show()

---
## 9. What We Covered

| Topic | Polars equivalent | DuckDB / SQL |
|-------|-------------------|--------------|
| Connect to engine | — | `duckdb.connect(':memory:')` |
| Run a query | — | `con.execute("SQL").df()` |
| Pick columns | `.select(["a", "b"])` | `SELECT a, b` |
| Filter rows | `.filter(pl.col("x") > 100)` | `WHERE x > 100` |
| Group + aggregate | `.group_by().agg()` | `GROUP BY` + `SUM() / AVG()` |
| Query existing DataFrame | — | `FROM df` (direct variable access) |
| Pass result to chart | `px.bar(df, ...)` | `px.bar(con.execute(query).df(), ...)` |

---

## 🔑 Key Takeaways

**DuckDB is not a replacement for Polars — it works with it.** The hybrid workflow is the point: Polars handles data transformations, DuckDB handles SQL queries on that same data, and Plotly handles the charts. All three together is what makes Project 1 possible.

**The only new syntax is SQL.** And the hardest part of SQL — the logic of filtering, grouping, selecting — I already knew from Polars. The adjustment was just learning to write it differently.

**`.df()` is the bridge.** Every DuckDB result comes back as a DataFrame when you call `.df()` at the end. That's what lets you pass it straight into Plotly, or do more Polars operations on it.

---

## ✅ Status

- `duckdb.connect()` — understood
- `con.execute(SQL)` — comfortable
- Querying Polars DataFrames directly — understood (Replacement Scans)
- Full pipeline: SQL query → Plotly chart — done

**Next up: Streamlit — turning this pipeline into an actual web app.**